# Gradient Boosting

## 1. Load the training data

In [72]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import ParameterSampler, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

# Find the data and preprocessing code from either notebook location.
working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'src' / 'preprocessing_pipeline.py').is_file()
     and (path / 'Dataset' / 'processed').is_dir()),
    None,
)

sys.path.insert(0, str(project_root))
from src.preprocessing_pipeline import TARGET, build_preprocessor

RANDOM_STATE = 42
BAD, GOOD = 0, 1
CLASS_NAMES = ['bad', 'good']

data_dir = project_root / 'Dataset' / 'processed'

In [73]:
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')[TARGET]

print(f'Training Dataset Rows :  {len(X_train)}')
print(f'Training dataset features count : {X_train.shape[1]}')
display(y_train.value_counts().reindex([BAD, GOOD]).set_axis(CLASS_NAMES).to_frame('count'))

Training Dataset Rows :  700
Training dataset features count : 21


,count
bad,210
good,490


## 2. Tune the model and threshold

Use the same five stratified folds for every setting. Compare cutoffs for predicting bad credit using the validation probabilities. Rank by mean bad-class F1, then by higher bad-class recall and lower F1 variation across folds.

In [74]:
# Create the baseline pipeline
baseline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),
])

# Setting up the parameter grid for hyperparameter tuning
param_grid = {
    'model__n_estimators': [50, 100, 150, 200],
    'model__learning_rate': [0.025, 0.05, 0.1],
    'model__max_depth': [1, 2, 3],
    'model__min_samples_leaf': [3, 8, 16, 24],
    'model__subsample': [0.75, 0.9, 1.0],
}

# Generate 48 random hyperparameter combinations from the param_grid. Use the same random state for reproducibility.
sampled_settings = ParameterSampler(param_grid, n_iter=48, random_state=RANDOM_STATE)
all_settings = [{}, *sampled_settings]

In [75]:
# Create the stratified cross validatoin folds, with total 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Create a list of the folds, so they can be reused for each parameter and threshold combination.
folds = list(cv.split(X_train, y_train))
thresholds = np.arange(20, 61, 5) / 100


# Test each parameter combination using the cross validation, and store probability of each input in the training is bad for each parameter combination.
validation_probabilities = []
for params in all_settings:
    candidate = clone(baseline).set_params(**params)

    probabilities = cross_val_predict(
        candidate, 
        X_train, 
        y_train, 
        cv=folds, 
        method='predict_proba',
    )

    validation_probabilities.append(probabilities[:, BAD])

In [76]:

# Now we evaluate both both the parameter combinations and thresholds. 

# results will store the evaluation results for each configuration and threshold combination
results = []

# Loops trough each parameter config, and the probabilities of each training row being bad.
for params, bad_probability in zip(all_settings, validation_probabilities):

    # Loops through each thresold
    for threshold in thresholds:
        predictions = np.where(bad_probability >= threshold, BAD, GOOD)
        fold_f1 = [
            f1_score(
                y_train.iloc[rows], 
                predictions[rows], 
                pos_label=BAD, 
                zero_division=0,
            )
            for _, rows in folds
        ]
        
        results.append({
            'Parameters': params,
            'Threshold': threshold,
            'Mean F1 (bad)': np.mean(fold_f1),
            'F1 SD': np.std(fold_f1),
            'Recall (bad)': recall_score(y_train, predictions, pos_label=BAD, zero_division=0),
        })

In [77]:
# Get the basline result with a threshold of 0.5
baseline_result = next(
    result for result in results
    if not result['Parameters'] and result['Threshold'] == 0.5
)

# Sort the performance based on the mean F1, then recall, then F1 standard deviation
ranked_results = pd.DataFrame(results).sort_values(
    ['Mean F1 (bad)', 'Recall (bad)', 'F1 SD'],
    ascending=[False, False, True],
)


best_result = ranked_results.iloc[0]
best_params = best_result['Parameters']
best_threshold = best_result['Threshold']

In [78]:
pd.DataFrame([baseline_result, best_result.to_dict()], index=['Baseline', 'Selected'],).drop(columns='Parameters').round(4)

,Threshold,Mean F1 (bad),F1 SD,Recall (bad)
Baseline,0.50,0.5076,0.0578,0.4286
Selected,0.35,0.6056,0.0596,0.6762


In [79]:
pd.Series(best_params, name='Selected value', dtype=object).to_frame()

,Selected value
model__subsample,0.9
model__n_estimators,200
model__min_samples_leaf,24
model__max_depth,1
model__learning_rate,0.1


In [80]:
best_model = clone(baseline).set_params(**best_params).fit(X_train, y_train)

## 3. Evaluate on the held-out test set

In [81]:
# Load the untouched test set
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_test = pd.read_csv(data_dir / 'y_test.csv')[TARGET]

baseline_prediction = baseline.fit(X_train, y_train).predict(X_test)

# Find the probability column for bad credit before applying the cutoff.
bad_column = list(best_model.classes_).index(BAD)
bad_probability = best_model.predict_proba(X_test)[:, bad_column]
y_pred = np.where(bad_probability >= best_threshold, BAD, GOOD)

In [82]:
rows = []

for prediction in (baseline_prediction, y_pred):
    rows.append({
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision (bad)': precision_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'Recall (bad)': recall_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'F1 (bad)': f1_score(y_test, prediction, pos_label=BAD, zero_division=0),
    })

display(pd.DataFrame(rows, index=['Baseline', 'Selected']).round(4))

,Accuracy,Precision (bad),Recall (bad),F1 (bad)
Baseline,0.7833,0.7119,0.4667,0.5638
Selected,0.7700,0.6040,0.6778,0.6387


In [83]:
print('Confusion matrix for the selected model')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[BAD, GOOD]),
    index=pd.Index(CLASS_NAMES, name='Actual'),
    columns=pd.Index(CLASS_NAMES, name='Predicted'),
))

Confusion matrix for the selected model


Predicted,bad,good
Actual,,
bad,61,29
good,40,170


In [84]:
print(classification_report(
    y_test, y_pred, labels=[BAD, GOOD], target_names=CLASS_NAMES,
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

         bad     0.6040    0.6778    0.6387        90
        good     0.8543    0.8095    0.8313       210

    accuracy                         0.7700       300
   macro avg     0.7291    0.7437    0.7350       300
weighted avg     0.7792    0.7700    0.7735       300

